In [ ]:
# ================================
# STEP 1: Upload Excel files
# ================================
from google.colab import files
uploaded = files.upload()

# ================================
# STEP 2: Imports
# ================================
import pandas as pd
import re
from datetime import datetime

# ================================
# STEP 3: Parse filenames & sort by month
# ================================
file_info = []

for filename in uploaded.keys():
    if not filename.lower().endswith(".xlsx"):
        continue

    match = re.search(r'(\d{2}-[A-Za-z]{3}-\d{4})', filename)
    if not match:
        raise ValueError(f"❌ Month not found in filename: {filename}")

    month_date = datetime.strptime(match.group(1), "%d-%b-%Y")
    file_info.append((month_date, filename))

if not file_info:
    raise ValueError("❌ No valid Excel files uploaded")

file_info.sort(key=lambda x: x[0])  # Jan → Dec

# ================================
# STEP 4: Validate hotel name (A2)
# ================================
hotel_names = set()

for _, filename in file_info:
    df = pd.read_excel(filename, header=None)
    hotel = str(df.iloc[1, 0]).strip()  # A2

    if not hotel or hotel.lower() == "nan":
        raise ValueError(f"❌ Hotel name missing in A2: {filename}")

    hotel_names.add(hotel)

if len(hotel_names) != 1:
    raise ValueError(
        "❌ Hotel name mismatch across files:\n" +
        "\n".join(hotel_names)
    )

hotel_name = hotel_names.pop()

# ================================
# STEP 5: Extract rows 1–4 per month
# ================================
monthly_blocks = []

for month_date, filename in file_info:
    df = pd.read_excel(filename, header=None)

    if len(df) < 4:
        raise ValueError(f"❌ File has less than 4 rows: {filename}")

    block = df.iloc[0:4].copy()  # Excel rows 1–4

    # Write month label into column A of first row in block
    block.iloc[0, 0] = month_date.strftime("%b-%Y")

    monthly_blocks.append(block)

# ================================
# STEP 6: Stack all months
# ================================
final_df = pd.concat(monthly_blocks, ignore_index=True)

# ================================
# STEP 7: Export
# ================================
safe_hotel = hotel_name.replace(" ", "_").replace("/", "_")
output_file = f"{safe_hotel}_Forecast_Jan_to_Dec_Extract.xlsx"

final_df.to_excel(output_file, index=False, header=False)

files.download(output_file)
